# 05 · Public Bitcoin Mining Companies

**Purpose:** Equity analysis of all major US-listed BTC miners and BTC investment vehicles.

**Tickers covered:**
| Ticker | Company |
|--------|---------|
| MARA | Marathon Digital Holdings |
| RIOT | Riot Platforms |
| CLSK | CleanSpark |
| IREN | Iris Energy |
| HUT | Hut 8 Corp |
| BTBT | Bit Digital |
| CIFR | Cipher Mining |
| CORZ | Core Scientific |
| WULF | TeraWulf |
| BITF | Bitfarms |

**Plus ETFs:** IBIT, FBTC, BITO (BTC ETFs), WGMI, SATO (mining ETFs), MSTR (BTC proxy)

---

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.data.yfinance_fetcher import (
    fetch_prices, fetch_ohlcv, fetch_fundamentals, compute_returns
)
from src.models.mining_metrics import (
    compute_all_betas, realized_vol, sharpe_ratio,
    mining_sector_snapshot, MINER_BTC_TREASURY, MINER_HASHRATE_PH
)
from src.utils.plotting import (
    plot_normalized_returns, plot_correlation_heatmap, plot_beta_vs_vol,
    PLOTLY_TEMPLATE, BTC_ORANGE, MINER_COLORS
)
from config import MINERS, BTC_ETFS

pd.options.display.float_format = '{:,.3f}'.format
print('Setup complete.')

## 1. Fetch Price Data

In [ ]:
all_tickers = list(MINERS.keys()) + list(BTC_ETFS.keys()) + ['BTC-USD']

prices = fetch_prices(all_tickers, period='2y')
prices = prices.dropna(how='all')

print(f'Data loaded: {prices.shape[1]} tickers, {len(prices)} trading days')
print(f'Date range: {prices.index[0].date()} → {prices.index[-1].date()}')

# Report available tickers
available = prices.columns.tolist()
missing = [t for t in all_tickers if t not in available]
if missing:
    print(f'Missing tickers: {missing}')

btc_price = prices['BTC-USD']
miner_prices = prices[list(MINERS.keys())].dropna(how='all', axis=1)
miner_prices.tail(3)

## 2. Normalized Performance (Base = 100)

In [ ]:
# All miners + BTC for the past 1 year
one_year_ago = pd.Timestamp.today() - pd.DateOffset(years=1)
fig = plot_normalized_returns(
    prices[miner_prices.columns.tolist() + ['BTC-USD']],
    base_date=one_year_ago.strftime('%Y-%m-%d'),
    title='Mining Stocks vs BTC — 1-Year Normalized Returns (Base=100)',
)
fig.show()

In [ ]:
# Performance summary table
returns_1y = prices.loc[prices.index >= one_year_ago].pct_change().dropna()

perf = pd.DataFrame({
    '1Y Return': (prices.iloc[-1] / prices[prices.index >= one_year_ago].iloc[0] - 1) * 100,
    'YTD Return': (prices.iloc[-1] / prices[prices.index >= f'{prices.index[-1].year}-01-01'].iloc[0] - 1) * 100,
    '30d Return': (prices.iloc[-1] / prices.iloc[-22] - 1) * 100,
}).round(1)

perf = perf.loc[list(MINERS.keys()) + ['BTC-USD']].dropna()
perf.columns = ['1Y %', 'YTD %', '30d %']
print('Performance Summary:')
print(perf.sort_values('1Y %', ascending=False).to_string())

## 3. Beta to BTC

In [ ]:
returns = compute_returns(prices)

beta_df = compute_all_betas(returns, btc_col='BTC-USD', window=90)

print('90-day OLS Beta to BTC:')
display_cols = ['beta', 'alpha_annualized', 'r_squared', 't_stat_beta']
print(beta_df[display_cols].sort_values('beta', ascending=False).round(3).to_string())

In [ ]:
# Bar chart of betas
miner_betas = beta_df.loc[beta_df.index.isin(list(MINERS.keys()))].sort_values('beta', ascending=True)

fig = go.Figure(go.Bar(
    x=miner_betas['beta'],
    y=miner_betas.index,
    orientation='h',
    marker_color=[MINER_COLORS.get(t, BTC_ORANGE) for t in miner_betas.index],
))
fig.add_vline(x=1.0, line_dash='dash', line_color='white',
               annotation_text='β=1 (BTC-like)', annotation_position='top right')
fig.update_layout(
    title='Mining Stocks: 90-Day Beta to BTC',
    xaxis_title='Beta to BTC', yaxis_title='Ticker',
    template=PLOTLY_TEMPLATE, height=450,
)
fig.show()

## 4. Realized Volatility

In [ ]:
rv_30d = realized_vol(returns, window=30).iloc[-1]
rv_30d_pct = (rv_30d * 100).rename('30d_RV_pct')

miner_rv = rv_30d_pct.loc[rv_30d_pct.index.isin(list(MINERS.keys()) + ['BTC-USD'])].sort_values(ascending=False)

fig = go.Figure(go.Bar(
    x=miner_rv.index,
    y=miner_rv.values,
    marker_color=[MINER_COLORS.get(t, '#888') for t in miner_rv.index],
))
fig.update_layout(
    title='Realized Volatility (30-day Annualized)',
    yaxis_title='Annualized Vol (%)',
    template=PLOTLY_TEMPLATE, height=400,
)
fig.show()
print(miner_rv.to_string())

## 5. Correlation Matrix

In [ ]:
tickers_for_corr = list(MINERS.keys()) + ['MSTR', 'WGMI', 'BTC-USD']
tickers_for_corr = [t for t in tickers_for_corr if t in returns.columns]

fig = plot_correlation_heatmap(
    returns[tickers_for_corr].dropna(),
    title='Return Correlations: Miners, ETFs & BTC (Daily, 2Y)'
)
fig.show()

## 6. Beta vs Volatility Scatter

In [ ]:
sector_vol = rv_30d_pct.to_frame('vol_30d')

fig = plot_beta_vs_vol(
    beta_df,
    sector_vol,
    title='Bitcoin Miners: Beta to BTC vs Realized Volatility',
)
fig.show()

## 7. BTC Treasury Holdings

In [ ]:
current_btc_price = float(btc_price.iloc[-1])

treasury_data = [
    {'ticker': t, 'btc_held': btc, 'usd_value_M': btc * current_btc_price / 1e6}
    for t, btc in MINER_BTC_TREASURY.items() if btc > 0
]
treasury_df = pd.DataFrame(treasury_data).sort_values('btc_held', ascending=False)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['BTC Held (coins)', 'BTC Treasury Value ($M)'])

fig.add_trace(go.Bar(
    x=treasury_df['ticker'], y=treasury_df['btc_held'],
    marker_color=[MINER_COLORS.get(t, BTC_ORANGE) for t in treasury_df['ticker']],
    name='BTC Held',
), row=1, col=1)

fig.add_trace(go.Bar(
    x=treasury_df['ticker'], y=treasury_df['usd_value_M'],
    marker_color=[MINER_COLORS.get(t, '#3498DB') for t in treasury_df['ticker']],
    name='USD Value ($M)',
), row=1, col=2)

fig.update_layout(
    title=f'Bitcoin Treasury Holdings (at ${current_btc_price:,.0f}/BTC)',
    template=PLOTLY_TEMPLATE, height=400, showlegend=False,
)
fig.show()
print(treasury_df.set_index('ticker').round(1).to_string())

## 8. Installed Hashrate Comparison

In [ ]:
hr_data = [
    {'ticker': t, 'hashrate_EHs': ph / 1e6, 'hashrate_ph': ph}
    for t, ph in MINER_HASHRATE_PH.items()
]
hr_df = pd.DataFrame(hr_data).sort_values('hashrate_EHs', ascending=False)

fig = go.Figure(go.Bar(
    x=hr_df['ticker'],
    y=hr_df['hashrate_EHs'],
    marker_color=[MINER_COLORS.get(t, BTC_ORANGE) for t in hr_df['ticker']],
    text=[f"{v:.1f} EH/s" for v in hr_df['hashrate_EHs']],
    textposition='outside',
))
fig.update_layout(
    title='Installed Hashrate by Public Miner (EH/s)',
    yaxis_title='Hashrate (EH/s)',
    template=PLOTLY_TEMPLATE, height=400,
)
fig.show()

## 9. Sector Snapshot Table

In [ ]:
try:
    fundamentals = fetch_fundamentals(list(MINERS.keys()))
except Exception as e:
    print(f'Fundamentals fetch failed: {e}. Using price-only snapshot.')
    fundamentals = None

snapshot = mining_sector_snapshot(prices, current_btc_price, fundamentals)
print('\nMining Sector Snapshot:')
print(snapshot.to_string())

## 10. ETF Analysis: WGMI, SATO vs BTC

In [ ]:
etf_tickers = [t for t in list(BTC_ETFS.keys()) if t in prices.columns]
etf_tickers += ['BTC-USD']

if len(etf_tickers) > 1:
    fig = plot_normalized_returns(
        prices[etf_tickers],
        base_date=one_year_ago.strftime('%Y-%m-%d'),
        title='BTC ETFs & Mining ETFs vs BTC (1Y Normalized)',
    )
    fig.show()

    etf_betas = beta_df.loc[beta_df.index.isin(etf_tickers)][['beta', 'r_squared']].round(3)
    print('ETF Betas to BTC:')
    print(etf_betas.to_string())
else:
    print('Limited ETF data available.')

## Summary

- Mining stocks have **high beta to BTC** (typically 1.5–3.0x) and even higher realized vol
- **MARA and RIOT** have the largest BTC treasuries, making them partly BTC holding companies
- **WGMI** and **SATO** provide diversified mining exposure in ETF form
- **MSTR** is an extreme BTC proxy with 400%+ BTC beta due to leverage
- Correlation within the mining sector is very high (>0.85) — they largely move together

**Next:** [06 · Portfolio Factor Model →](./06_portfolio_factor_model.ipynb)